### setup.sh

No script setup.sh , revise os comandos. O programa irá:

    - Crie uma conta de armazenamento no seu grupo de recursos do Azure.

    - Faça o upload de arquivos da sua pasta local `sampleforms` para um contêiner chamado `sampleforms` na conta de armazenamento.

    - Imprimir um URI de assinatura de acesso compartilhado

In [ ]:
#!/bin/bash

# Set variable values
subscription_id="YOUR_SUBSCRIPTION_ID"
resource_group="YOUR_RESOURCE_GROUP"
location="YOUR_LOCATION_NAME"
expiry_date="2026-01-01T00:00:00Z"

# Get random numbers to create unique resource names
unique_id=$((1 + RANDOM % 99999))

# Create a storage account in your Azure resource group
echo "Creating storage..."
az storage account create --name "ai102form$unique_id" --subscription "$subscription_id" --resource-group "$resource_group" --location "$location" --sku Standard_LRS --encryption-services blob --default-action Allow --allow-blob-public-access true --only-show-errors --output none

echo "Uploading files..."
# Get storage key to create a container in the storage account
key_json=$(az storage account keys list --subscription "$subscription_id" --resource-group "$resource_group" --account-name "ai102form$unique_id" --query "[?keyName=='key1'].{keyName:keyName, permissions:permissions, value:value}")
key_string=$(echo "$key_json" | jq -r '.[0].value')
AZURE_STORAGE_KEY=${key_string}

# Create a container
az storage container create --account-name "ai102form$unique_id" --name sampleforms --public-access blob --auth-mode key --account-key "$AZURE_STORAGE_KEY" --output none

# Upload files from your local sampleforms folder to a container called sampleforms in the storage account
# Each file is uploaded as a blob
az storage blob upload-batch -d sampleforms -s ./sample-forms --account-name "ai102form$unique_id" --auth-mode key --account-key "$AZURE_STORAGE_KEY" --output none

# Set a variable value for future use
STORAGE_ACCT_NAME="ai102form$unique_id"

# Get a Shared Access Signature (a signed URI that points to one or more storage resources) for the blobs in sampleforms
SAS_TOKEN=$(az storage container generate-sas --account-name "ai102form$unique_id" --name sampleforms --expiry "$expiry_date" --permissions rwl --auth-mode key --account-key "$AZURE_STORAGE_KEY")
URI="https://$STORAGE_ACCT_NAME.blob.core.windows.net/sampleforms?$SAS_TOKEN"

# Print the generated Shared Access Signature URI, which is used by Azure Storage to authorize access to the storage resource
echo "-------------------------------------"
echo "SAS URI: $URI"

## Treine o modelo usando o Document Intelligence Studio.

https://documentintelligence.ai.azure.com/studio

Treine o modelo usando o Document Intelligence Studio.
Agora você irá treinar o modelo usando os arquivos carregados na conta de armazenamento.

Abra uma nova aba do navegador e acesse o Document Intelligence Studio em https://documentintelligence.ai.azure.com/studio.
Desça até a seção Modelos personalizados e selecione o bloco Modelo de extração personalizado .
Se solicitado, faça login com suas credenciais do Azure.
Se for perguntado qual recurso do Azure AI Document Intelligence usar, selecione a assinatura e o nome do recurso que você usou ao criar o recurso do Azure AI Document Intelligence.
Em Meus Projetos , crie um novo projeto com a seguinte configuração:

Insira os detalhes do projeto :
Nome do projeto : Um nome válido para o seu projeto
Configurar recurso de serviço :
Assinatura : Sua assinatura do Azure
Grupo de recursos : O grupo de recursos onde você implantou seu recurso de Inteligência de Documentos.
Recurso de inteligência de documentos Seu recurso de inteligência de documentos (selecione a opção Definir como padrão e use a versão padrão da API)
Conectar a fonte de dados de treinamento :
Assinatura : Sua assinatura do Azure
Grupo de recursos : O grupo de recursos onde você implantou seu recurso de Inteligência de Documentos.
Conta de armazenamento : A conta de armazenamento criada pelo script de instalação (selecione a opção "Definir como padrão" , selecione o sampleformscontêiner de blobs e deixe o caminho da pasta em branco).
Após criar seu projeto, no canto superior direito da página, selecione Treinar para treinar seu modelo. Utilize as seguintes configurações:
ID do modelo : Um nome válido para o seu modelo ( você precisará do nome do ID do modelo na próxima etapa ).
Modo de compilação : Modelo.
Selecione Ir para Modelos .
O treinamento pode levar algum tempo. Aguarde até que o status seja " concluído com sucesso" .


## Teste seu modelo personalizado de Inteligência de Documentos

In [ ]:
# requirements

# pip install python-dotenv
# pip install azure-ai-formrecognizer==3.3.3

### test-model.py

In [ ]:
from azure.core.credentials import AzureKeyCredential
from azure.ai.formrecognizer import DocumentAnalysisClient
from dotenv import load_dotenv
import os


def main():

    # Clear the console
    os.system('cls' if os.name=='nt' else 'clear')

    try:
        # Get configuration settings 
        load_dotenv()
        endpoint = os.getenv("DOC_INTELLIGENCE_ENDPOINT")
        key = os.getenv("DOC_INTELLIGENCE_KEY")
        model_id = os.getenv("MODEL_ID")

        formUrl = "https://github.com/MicrosoftLearning/mslearn-ai-information-extraction/blob/main/Labfiles/custom-doc-intelligence/test1.jpg?raw=true"

        document_analysis_client = DocumentAnalysisClient(
            endpoint=endpoint, credential=AzureKeyCredential(key)
        )

        # Make sure your document's type is included in the list of document types the custom model can analyze
        response = document_analysis_client.begin_analyze_document_from_url(model_id, formUrl)
        result = response.result()

        for idx, document in enumerate(result.documents):
            print("--------Analyzing document #{}--------".format(idx + 1))
            print("Document has type {}".format(document.doc_type))
            print("Document has confidence {}".format(document.confidence))
            print("Document was analyzed by model with ID {}".format(result.model_id))
            for name, field in document.fields.items():
                field_value = field.value if field.value else field.content
                print("Found field '{}' with value '{}' and with confidence {}".format(name, field_value, field.confidence))

        print("-----------------------------------")
    except Exception as ex:
        print(ex)

    print("\nAnalysis complete.\n")

if __name__ == "__main__":
    main()     